# Inputs

In [ ]:
# bd.projects.set_current(f"{list(bd.projects)[1]}")
# bd.projects.delete_project("case_study", delete_dir=True)

## libraries

In [ ]:
# python general
import os
import time
from peewee import DoesNotExist
import pandas as pd
import matplotlib.pyplot as plt

# brightway: lca
import bw2data as bd
from bw2data.errors import UnknownObject

# local modules: circularity and lca
from circularity_lci.burden_free_analyzer import BurdenFreeAnalyzer
from circularity_lci.biosphere_flow_manager import BiosphereFlowManager
from circularity_lci.circularity_calculator import CircularityCalculator

## input data

In [ ]:
# ecoinvent
ei_username = os.getenv("EI_USERNAME")
ei_password = os.getenv("EI_PASSWORD")

excluded_flows = [
    "BOD5, Biological Oxygen Demand",
    "COD, Chemical Oxygen Demand",
    "DOC, Dissolved Organic Carbon",
    "TOC, Total Organic Carbon"
]

# valuable compartment for water flows, see function file for usability
valuable_water_compartments = ['ground-', 'surface water'] # for water flows
exclude_water = True  # Set to False if water flows are included # should be based on wet_mass or dry_mass => check dry_mass working
mass_strategy = "full_mass"      # options: "full_mass", "water_mass", "dry_mass"
energy_strategy = "full_energy"

# fit w<ith functions.py compliancies
project_name = "case_study" 
database_provider = "ecoinvent"  
database_version = "3.8"
database_systemmodel = "cutoff"
ecospold_folder = None
technosphere_db_name = f"{database_provider}-{database_version}-{database_systemmodel}"
biosphere_db_name = f"{database_provider}-{database_version}-biosphere"

cutoff_upr_name = "copper_cathode_cutoff_upr" # define a name for your cutoff input


## brightway project init

In [ ]:
# --- 1. Initialize Burden-Free Analyzer for the project ---

# You can also start directly from an existing project made of different calculation set-ups.
# list(bd.projects)
# bd.projects.set_current("your_project_name")

burden_free_analyzer = BurdenFreeAnalyzer(
    project_name=project_name,
    database_provider=database_provider, 
    ecospold_folder=ecospold_folder,  
    technosphere_db_name=technosphere_db_name,  
    database_version=database_version,  
    database_systemmodel=database_systemmodel  
)

burden_free_analyzer.setup_project()

# 2 scenarios modelling (baseline, improved)

In [ ]:
def create_copper_cathode_cutoff_activity(technosphere_db_name='database_name', activity_code = 'act_code'):
    """
    Creates a new activity in the ecoinvent-3.8-cutoff database with the code 'copper_cathode_cutoff'.
    Deletes any existing activity with the same code first, then creates a new activity with specified
    attributes and a production exchange. Forces database reprocessing to make the activity available.
    Prints confirmation upon success.
    """
    # Database and activity details
    activity_name = "copper cathode, cut-off unit process"
    location = "GLO"
    unit = "kilogram"
    product = "copper cathode, cut-off product flow"
    description = "This is an artificial cut-off unit process, that furnished copper for cathodes"

    # Access the database
    db = bd.Database(technosphere_db_name)

    # Delete existing activity if present
    try:
        existing_activity = db.get(activity_code) # type: ignore
        existing_activity.delete()
        print(f"Deleted existing activity with code '{activity_code}'")
    except (UnknownObject, DoesNotExist):
        print(f"No existing activity with code '{activity_code}' found. Proceeding to create a new one.")

    # Create new activity
    act = db.new_activity( # type: ignore
        code=activity_code,
        name=activity_name,
        location=location,
        unit=unit,
        product=product,
        reference_product=product,
        description=description
    )

    # Create and save production exchange
    act.new_exchange(
        input=(technosphere_db_name, activity_code),
        output=(technosphere_db_name, activity_code),
        amount=1.0,
        type="production",
        unit=unit,
        name=product,
        product=product,
        reference_product=product
    ).save()

    # Save activity and process database
    act.save()
    db.process()

    print(f"Successfully created activity '{activity_code}' with reference product '{product}'")

def copy_activity_simple(original_activity, new_code, technosphere_db_name):
    """Simple activity copying using Brightway2's internal methods"""
    
    # Get the database
    db = bd.Database(technosphere_db_name)
    
    # Delete existing if present - CORRECTED
    existing_activities = [act for act in db if act.get('code') == new_code] # type: ignore
    if existing_activities:
        print(f"Deleting existing activity with code: {new_code}")
        existing_activities[0].delete()
    
    # Use Brightway2's built-in copy functionality
    new_activity = original_activity.copy(new_code)
    new_activity['description'] = f"{original_activity.get('description', '')} Modified copy"
    new_activity.save()
    
    return new_activity

def replace_exc_proportion_by_another_upr(activity, original_activity, original_exc_key, new_process_key, proportion=0.5):
    """
    Replace an exchange proportionally with another UPR process
    Uses the original amount from the unmodified activity
    """
    # Get just the code part from the new_process_key
    new_process_code = new_process_key[1]
    
    # First, find the original amount from the unmodified activity
    original_amount = 0
    for exc in original_activity.exchanges():
        if exc.input.key[1] == original_exc_key:
            original_amount = exc.amount
            break
    
    print(f"Original amount from unmodified activity: {original_amount}")
    
    # Delete ALL existing exchanges for both processes from the copied activity
    exchanges_to_delete = []
    for exc in activity.exchanges():
        if exc.input.key[1] == original_exc_key or exc.input.key[1] == new_process_code:
            exchanges_to_delete.append(exc)
    
    # Delete all related exchanges
    for exc in exchanges_to_delete:
        exc.delete()
    
    print(f"Deleted {len(exchanges_to_delete)} existing exchanges from copied activity")
    
    # Only proceed if we found the original exchange
    if original_amount > 0:
        # Calculate new amounts based on the original unmodified amount
        new_original_amount = original_amount * (1-proportion)
        new_upr_amount = original_amount * proportion
        
        # Create the modified exchange for original process (50%)
        activity.new_exchange(
            input=(activity['database'], original_exc_key),  # Original process
            amount=new_original_amount,
            unit='kilogram',  # Assuming kilogram, adjust if needed
            type="technosphere",
        ).save()
        
        # Create the new exchange for cut-off UPr (50%)
        activity.new_exchange(
            input=new_process_key,
            amount=new_upr_amount,
            unit='kilogram',  # Assuming kilogram, adjust if needed
            type="technosphere",
        ).save()
        
        print(f"Created new exchanges: {new_original_amount} for original, {new_upr_amount} for UPR")
    else:
        print("No original exchange found in unmodified activity")


In [ ]:
# Execute the function
create_copper_cathode_cutoff_activity(technosphere_db_name, cutoff_upr_name)

### anode production processes to modify ###
# the processes
original_anode_cn = bd.get_activity((technosphere_db_name, 'be6c303c552a3e3c5a3a2332842ee39e')) # such for this code in activity_browser (easiest way for me)
original_anode_row = bd.get_activity((technosphere_db_name, 'c727d040313885163dfa52b91b1b8b7a'))
# copy them
anode_cn_copy = copy_activity_simple(original_anode_cn, "anode_cn_copy", technosphere_db_name)
anode_row_copy = copy_activity_simple(original_anode_row, "anode_row_copy_copy", technosphere_db_name)
# Usage - pass both the copied activity AND the original unmodified activity
print("Modifying anode_cn_copy...")
replace_exc_proportion_by_another_upr(anode_cn_copy, original_anode_cn, '4f669c71cf22f6b782575a3ed1d34807', (technosphere_db_name, "copper_cathode_cutoff_upr"), 0.5)
print("\nModifying anode_row_copy...")
replace_exc_proportion_by_another_upr(anode_row_copy, original_anode_row, '4f669c71cf22f6b782575a3ed1d34807', (technosphere_db_name, "copper_cathode_cutoff_upr"), 0.5)

### market for anode to modify ###
# copy the market for anode glo
anode_market_glo = bd.get_activity((technosphere_db_name, '2d99cac67854cb212704044fd550a434'))
print("Copying anode_market_glo...")
anode_market_glo_copy = copy_activity_simple(anode_market_glo, "anode_market_glo_copy", technosphere_db_name)
print("Modifying anode_market_glo_copy...")
replace_exc_proportion_by_another_upr(anode_market_glo_copy, anode_market_glo, 'be6c303c552a3e3c5a3a2332842ee39e', anode_cn_copy.key, 1)
replace_exc_proportion_by_another_upr(anode_market_glo_copy, anode_market_glo, 'c727d040313885163dfa52b91b1b8b7a', anode_row_copy.key, 1)

### battery cell production to modify ###
# copy the battery cell production row
battery_cell_prod_LiBs_row = bd.get_activity((technosphere_db_name, '4d29fc61b64f4a197d0d8140c50303c1'))
print("Copying battery_cell_prod_LiBs_row...")
battery_cell_prod_LiBs_row_copy = copy_activity_simple(battery_cell_prod_LiBs_row, "battery_cell_prod_LiBs_row_copy", technosphere_db_name)
print("Modifying battery_cell_prod_LiBs_row_copy...")
replace_exc_proportion_by_another_upr(battery_cell_prod_LiBs_row_copy, battery_cell_prod_LiBs_row, '2d99cac67854cb212704044fd550a434', anode_market_glo_copy.key, 1)
# copy the battery cell production cn
battery_cell_prod_LiBs_cn = bd.get_activity((technosphere_db_name, 'b721b9e1c919e8fb2ba12f6ae67d2c1d'))
print("Copying battery_cell_prod_LiBs_cn...")
battery_cell_prod_LiBs_cn_copy = copy_activity_simple(battery_cell_prod_LiBs_cn, "battery_cell_prod_LiBs_cn_copy", technosphere_db_name)
print("Modifying battery_cell_prod_LiBs_cn_copy...")
replace_exc_proportion_by_another_upr(battery_cell_prod_LiBs_cn_copy, battery_cell_prod_LiBs_cn, '2d99cac67854cb212704044fd550a434', anode_market_glo_copy.key, 1)

### market for battery cell, Li-ion to modify ###
cell_LiBs_market_glo = bd.get_activity((technosphere_db_name, 'b2feecd5152754c08303bc84dc371b68'))
print("Copying cell_LiBs_market_glo...")
cell_LiBs_market_glo_copy = copy_activity_simple(cell_LiBs_market_glo, "cell_LiBs_market_glo_copy", technosphere_db_name)
print("Modifying cell_LiBs_market_glo_copy...")
replace_exc_proportion_by_another_upr(cell_LiBs_market_glo_copy, cell_LiBs_market_glo, 'b721b9e1c919e8fb2ba12f6ae67d2c1d', battery_cell_prod_LiBs_cn_copy.key, 1) # CN
replace_exc_proportion_by_another_upr(cell_LiBs_market_glo_copy, cell_LiBs_market_glo, '4d29fc61b64f4a197d0d8140c50303c1', battery_cell_prod_LiBs_row_copy.key, 1) # RoW

### battery production, Li-ion, rechargeable, prismatic ###
LiBs_tot_glo = bd.get_activity((technosphere_db_name, '3377a1ab4d9266e104f29c12b4443f31'))
print("Copying cell_LiBs_market_glo...")
LiBs_tot_glo_copy = copy_activity_simple(LiBs_tot_glo, "LiBs_tot_glo_copy", technosphere_db_name)
print("Modifying LiBs_tot_glo_copy...")
replace_exc_proportion_by_another_upr(LiBs_tot_glo_copy, LiBs_tot_glo, 'b2feecd5152754c08303bc84dc371b68', cell_LiBs_market_glo_copy.key, 1)



# Circularity Calculs

In [ ]:
# ⚠️ should be 97 flows for ecoinvent 3.8 since copper cathod resource input technosphere functional flow has been artificially created ⚠️
# --- 2. Analyze Burden-Free Activities --- 
burden_free_activities = burden_free_analyzer.analyze_all_databases()

# --- 3. Initialize Biosphere Flow Manager ---
biosphere_flow_manager = BiosphereFlowManager(biosphere_db_name)  # Your biosphere DB

# --- 4. Process Burden-Free Activities (Create Biosphere Flows) ---
biosphere_flow_manager.process(burden_free_activities)

In [ ]:
# Define the calculation setup
setup_name_with50Ri = "Improved_case"
setup_name_control = "Baseline_case"
bd.calculation_setups[setup_name_control] = {
    "inv": [
        {(technosphere_db_name, "3377a1ab4d9266e104f29c12b4443f31"): 454.0},
    ],
    "ia": [
        (
            "ReCiPe 2016 v1.03, midpoint (E)",
            "eutrophication: marine",
            "marine eutrophication potential (MEP)",
        )
    ],
    "description": "Controlled setup, LiBs craddle to gate",
}
bd.calculation_setups[setup_name_with50Ri] = {
    "inv": [
        {(technosphere_db_name, "LiBs_tot_glo_copy"): 454.0},
    ],
    "ia": [
        (
            "ReCiPe 2016 v1.03, midpoint (E)",
            "eutrophication: marine",
            "marine eutrophication potential (MEP)",
        )
    ],
    "description": "Improved Case, LiBs craddle to gate",
}


## results csv

In [ ]:
# --- 1. Initialize Circularity Calculator ---
circularity_calculator = CircularityCalculator(
    project_name=bd.projects.current,
    excluded_flows=excluded_flows,  
    valuable_water_compartments=valuable_water_compartments,
    exclude_water=exclude_water, # Set to False if water flows are included in the circularity analysis
    technosphere_db_name= technosphere_db_name,
    biosphere_db_name=biosphere_db_name,  # Your biosphere DB
    mass_strategy= mass_strategy,  # or "water_mass", "dry_mass"
    energy_strategy=energy_strategy  # or "renewable_energy", "non-renewable_energy"
)

# --- 2. Loop over all calculation setups ---
setup_names = list(bd.calculation_setups.keys())
results = {}  # Store results for each setup

for setup_name in setup_names:
    print(f"\n--- Analyzing setup: {setup_name} ---")

    # --- 3. Get Inventory Flows ---
    flows_df = circularity_calculator.get_inventory_flows(setup_name)

    # --- 4. Compute Circularity Metrics ---
    efficiency_df, _, _, detailed_flows_df = (
        circularity_calculator.compute_circularity_efficiency_variables(flows_df, setup_name=setup_name)
    )

    emf_df, _, _ = circularity_calculator.compute_circularity_EMF_indicators(flows_df, setup_name=setup_name)

    # --- 5. Store Results ---
    results[setup_name] = {
        "efficiency_df": efficiency_df,
        "emf_df": emf_df,
        "detailed_flows_df": detailed_flows_df
    }

    # --- 6. Print or Save Results ---
    output_dir = f"results/csv/{bd.projects.current}/{setup_name}"
    os.makedirs(output_dir, exist_ok=True)

    efficiency_df.to_csv(f"{output_dir}/efficiency_results.csv") # type: ignore
    emf_df.to_csv(f"{output_dir}/emf_results.csv") # type: ignore
    detailed_flows_df.to_csv(f"{output_dir}/detailed_flows.csv") # type: ignore

    print(f"Results saved to: {output_dir}")

## results plot

In [ ]:
# --- 7. Compare All Setups ---
# Define a function to safely extract a filename
def safe_filename(name):
    """Convert setup name into a safe filename."""
    return name.replace(" ", "_").replace(",", "").replace("(", "").replace(")", "")

def compare_circularity_setups_variables(results_df1, results_df2, setup_name1, setup_name2,
                                        mass_units=None, energy_units=None):
    """
    Enhanced comparison:
      - Builds a dataframe with per-unit, per-variable comparisons (absolute and % change).
      - Saves a CSV and plots side-by-side bar plots per unit for all common variables.
      - Also creates a focused plot for core circularity indicators (LFI, CFI, eta+, eta-).
    """
    if mass_units is None:
        mass_units = ['kilogram']
    if energy_units is None:
        energy_units = ['megajoule']

    output_dir = os.path.join("results", "comparison", f"{bd.projects.current}")
    os.makedirs(output_dir, exist_ok=True)
    safe1 = safe_filename(setup_name1)
    safe2 = safe_filename(setup_name2)

    # collect all units to compare
    units = []
    for u in mass_units + energy_units:
        if u in results_df1.index and u in results_df2.index:
            units.append(u)

    rows = []
    for unit in units:
        row1 = results_df1.loc[unit]
        row2 = results_df2.loc[unit]

        # variables to compare = union of columns present in both rows
        vars_union = sorted(set(row1.index) | set(row2.index))
        for var in vars_union:
            v1 = float(row1.get(var, 0.0)) if pd.notna(row1.get(var, None)) else 0.0
            v2 = float(row2.get(var, 0.0)) if pd.notna(row2.get(var, None)) else 0.0
            abs_diff = v2 - v1
            pct_change = None
            if v1 != 0:
                pct_change = (v2 - v1) / v1 * 100
            rows.append({
                "unit": unit,
                "variable": var,
                setup_name1: v1,
                setup_name2: v2,
                "abs_diff": abs_diff,
                "pct_change": pct_change
            })

    comparison_df = pd.DataFrame(rows)
    csv_path = os.path.join(output_dir, f"comparison.csv")
    comparison_df.to_csv(csv_path, index=False)
    print(f"✅ Comparison table saved: {csv_path}")

    # --- Plot per unit: grouped bar chart of variables ---
    for unit in sorted(set(comparison_df['unit'])):
        df_unit = comparison_df[comparison_df['unit'] == unit].set_index('variable')
        if df_unit.empty:
            continue

        # Keep top N variables by absolute value to avoid overcrowding (or plot all if few)
        df_unit['abs_max'] = df_unit[[setup_name1, setup_name2]].abs().max(axis=1)
        top_vars = df_unit.sort_values('abs_max', ascending=False).head(30).index  # cap to 30 variables
        plot_df = df_unit.loc[top_vars, [setup_name1, setup_name2]].fillna(0)

        ax = plot_df.plot(kind='bar', figsize=(14, 6))
        ax.set_title(f"Different calculation setups circularity variables per {unit}")
        ax.set_ylabel(unit)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt_path = os.path.join(output_dir, f"by_{unit}.png")
        plt.savefig(plt_path, dpi=300, bbox_inches='tight')
        plt.show()

        # --- Absolute difference and percent-change plots (added) ---
        # prepare diff / pct series for the same top variables
        try:
            diff_series = df_unit.loc[top_vars, 'abs_diff'].fillna(0) # v2 - v1
        except Exception:
            diff_series = (plot_df[setup_name2] - plot_df[setup_name1]).iloc[:,0] if not plot_df.empty else pd.Series(dtype=float)

        if diff_series.sum() != 0:
            figd, axd = plt.subplots(figsize=(14, 4))
            diff_series.plot(kind='bar', color='tab:red', ax=axd)
            axd.set_title(f"Absolute change compare to control with {unit}")
            axd.set_ylabel(unit)
            axd.axhline(0, color='black', linewidth=0.8)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            diff_path = os.path.join(output_dir, f"absdiff_{unit}.png")
            plt.savefig(diff_path, dpi=300, bbox_inches='tight')
            plt.show()
        else:
            print(f"Skipping absolute-diff plot for {unit} (all zeros).")

        # percent change plot (where available)
        if 'pct_change' in df_unit.columns:
            pct_series = df_unit.loc[top_vars, 'pct_change'].fillna(0)
            if pct_series.abs().sum() != 0:
                figp, axp = plt.subplots(figsize=(14, 4))
                pct_series.plot(kind='bar', color='tab:purple', ax=axp)
                axp.set_title(f"Percent change compare to control with unit {unit}")
                axp.set_ylabel("Percent change (%)")
                axp.axhline(0, color='black', linewidth=0.8)
                plt.xticks(rotation=45, ha='right')
                plt.tight_layout()
                pct_path = os.path.join(output_dir, f"pctchange_{unit}.png")
                plt.savefig(pct_path, dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print(f"Skipping percent-change plot for {unit} (no non-zero pct changes).")

    # --- Focused plot for core indicators (LFI, CFI, eta+, eta-) if present ---
    indicator_keywords = ['LFI', 'CFI', 'eta+', 'eta-', 'efficiency (eta+)', 'inefficiency (eta-)']
    df_ind = comparison_df[comparison_df['variable'].str.contains('|'.join([kw.replace('+','\\+') for kw in indicator_keywords]), case=False, regex=True)]
    if not df_ind.empty:
        # pivot to have variables as rows and setups as columns (aggregate across units by summing)
        pivot = df_ind.groupby('variable')[[setup_name1, setup_name2]].sum()
        pivot.plot(kind='bar', figsize=(12, 6))
        plt.title(f"Core circularity indicators summed across units calculation setups")
        plt.ylabel("Value (units aggregated)")
        plt.xticks(rotation=45, ha='right')
        focused_path = os.path.join(output_dir, f"core_indicators.png")
        plt.tight_layout()
        plt.savefig(focused_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"✅ Core indicators plot saved: {focused_path}")
    else:
        print("ℹ️ No core circularity indicators (LFI/CFI/eta) found in the comparison table.")

    return comparison_df

def compare_circularity_setups_variables_df(results_df1, results_df2, setup_name1, setup_name2,
                                        mass_units=None, energy_units=None):
    """
    Compare two circularity setups and return a DataFrame with the results.
    """
    if mass_units is None:
        mass_units = ['kilogram']
    if energy_units is None:
        energy_units = ['megajoule']

    # Collect all units to compare
    units = []
    for u in mass_units + energy_units:
        if u in results_df1.index and u in results_df2.index:
            units.append(u)

    rows = []
    for unit in units:
        row1 = results_df1.loc[unit]
        row2 = results_df2.loc[unit]

        # Variables to compare = union of columns present in both rows
        vars_union = sorted(set(row1.index) | set(row2.index))
        for var in vars_union:
            v1 = float(row1.get(var, 0.0)) if pd.notna(row1.get(var, None)) else 0.0
            v2 = float(row2.get(var, 0.0)) if pd.notna(row2.get(var, None)) else 0.0
            abs_diff = v2 - v1
            pct_change = None
            if v1 != 0:
                pct_change = (v2 - v1) / v1 * 100
            rows.append({
                "unit": unit,
                "variable": var,
                setup_name1: v1,
                setup_name2: v2,
                "abs_diff": abs_diff,
                "pct_change": pct_change
            })

    comparison_df = pd.DataFrame(rows)
    return comparison_df

# Compare each pair of setups
for i, setup_name1 in enumerate(setup_names):
    for j, setup_name2 in enumerate(setup_names):
        if i < j:  # Avoid comparing a setup with itself or repeating comparisons
            print(f"\n--- Comparing {setup_name1} vs {setup_name2} ---")

            # Extract results for comparison
            efficiency_df1 = results[setup_name1]["efficiency_df"]
            efficiency_df2 = results[setup_name2]["efficiency_df"]

            # Compare circularity setups
            comparison_df = compare_circularity_setups_variables(
                efficiency_df1, efficiency_df2, setup_name1, setup_name2,
                mass_units=["kilogram"], energy_units=["megajoule"]
            )

            # Save comparison results
            output_dir = f"results/comparison/{bd.projects.current}"
            os.makedirs(output_dir, exist_ok=True)

            safe1 = safe_filename(setup_name1)
            safe2 = safe_filename(setup_name2)
            comparison_df.to_csv(f"{output_dir}/comparison_{safe1}_vs_{safe2}.csv", index=False)

            print(f"Comparison results saved to: {output_dir}/comparison_{safe1}_vs_{safe2}.csv")

print("\n--- All analyses completed! ---")

In [ ]:
comp_df = compare_circularity_setups_variables_df(efficiency_df1, efficiency_df2, setup_name_control, setup_name_with50Ri)

print("\nFINAL comparison shape:", comp_df.shape)
# show full small table (or head if large)
with pd.option_context('display.max_rows', 200, 'display.max_columns', 200):
    display(comp_df)